# Phase 5 — Control Candidates

Builds the pool of never-retracted authors from which controls are matched.

**Input:** `data/interim/phase02_paper_level.csv`,
`data/interim/phase02_author_paper.csv`, the OpenAlex snapshot

**Outputs**

| File | Contents |
|---|---|
| `data/interim/phase05_candidates.csv` | one row per candidate author per paper |

## The sampling design

For every retracted paper the journal and publication year are known. Authors
who published in that same journal in that same year, and who never appear on a
retracted paper in this study, form the comparison pool: same venue, same
field, same era, same approximate quality bar.

**Journal-year sampling rather than co-authors.** A retracted author's
collaborators may share the consequences of the retraction through the
collaboration itself, so they cannot serve as untreated comparisons. Authors
drawn from the same journal-year have no network connection to the treated
author and no path for spillover.

**Exclusion is global, not per paper.** Anyone appearing anywhere in the
treated set is removed, so a control for one retraction cannot be a treated
author for another.

**Every eligible paper in each journal-year is retained.** Restricting to a
fixed number per journal-year would introduce an arbitrary cut, and the
journal-years where the pool is thinnest are precisely those where treated
authors are most likely to go unmatched at Phase 6.

This phase identifies candidates only. Matching happens next, and publication
histories are extracted for matched controls alone, which keeps the expensive
step as small as it can be.

In [1]:
import csv
import json
import os
import sys
import time

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, "src")

from snapshot import Snapshot, strip_id

PAPER_LEVEL = "data/interim/phase02_paper_level.csv"
AUTHOR_PAPER = "data/interim/phase02_author_paper.csv"
OUT_CANDIDATES = "data/interim/phase05_candidates.csv"

CANDIDATE_COLUMNS = ["author_id", "work_id", "source_id", "pub_year",
                     "position", "is_corresponding", "country"]

SCAN_COLUMNS = ["id", "publication_year", "is_retracted", "authorships",
                "primary_location"]

JY_MIN_YEAR, JY_MAX_YEAR = 2005, 2023

# Restrict the scan for testing. None runs the full snapshot.
LIMIT_FILES = None
SKIP_FILES = 0

pd.set_option("display.width", 200)
os.makedirs("data/interim", exist_ok=True)

snap = Snapshot()
d = snap.describe("works")
print(f"works: {d['files']:,} files, {d['bytes'] / 2**30:.0f} GiB")

works: 2,446 files, 675 GiB


## Journal-years to sample

In [2]:
papers = pd.read_csv(PAPER_LEVEL, low_memory=False)
jy = (papers.dropna(subset=["source_id", "pub_year"])[["source_id", "pub_year"]]
            .drop_duplicates())
jy["pub_year"] = jy.pub_year.astype(int)
jy = jy[(jy.pub_year >= JY_MIN_YEAR) & (jy.pub_year <= JY_MAX_YEAR)]

print(f"retracted papers   {len(papers):,}")
print(f"journal-years      {len(jy):,}")
print(f"distinct journals  {jy.source_id.nunique():,}")
print(f"years              {jy.pub_year.min()} to {jy.pub_year.max()}")

treated = set(pd.read_csv(AUTHOR_PAPER, usecols=["author_id"])
                .author_id.dropna().astype(str))
print(f"\ntreated authors to exclude  {len(treated):,}")

want_jy = set(zip(jy.source_id.astype(str), jy.pub_year))

retracted papers   17,591
journal-years      7,763
distinct journals  4,232
years              2005 to 2022

treated authors to exclude  63,758


## Extraction

Works are retained where the journal and publication year match a treated
paper's and the record is not itself retracted. The authorship list of each is
then flattened, excluding anyone in the treated set.

Rows are written as each batch is processed. The candidate pool runs to
millions of rows, and holding the source works in memory alongside it would be
wasteful when nothing downstream needs them.

In [3]:
PREFIX = "https://openalex.org/"
want_sources = pa.array(
    sorted({s for s, _ in want_jy} | {PREFIX + s for s, _ in want_jy}),
    type=pa.string())
want_years = pa.array(sorted({y for _, y in want_jy}), type=pa.int32())

treated_arr = pa.array(sorted(treated | {PREFIX + a for a in treated}),
                       type=pa.string())


class Extractor:
    """Writes candidate author rows as each batch is processed."""

    def __init__(self, path, columns):
        self.f = open(path, "w", newline="", encoding="utf-8")
        self.writer = csv.DictWriter(self.f, fieldnames=columns)
        self.writer.writeheader()
        self.n_works = 0
        self.n_rows = 0
        self.authors = set()

    @staticmethod
    def _source_ids(tbl):
        """primary_location.source.id. The id directly on primary_location
        identifies the location record rather than the venue."""
        loc = tbl.column("primary_location")
        if isinstance(loc, pa.ChunkedArray):
            loc = loc.combine_chunks()
        return pc.struct_field(loc, ["source", "id"])

    def handle(self, tbl, path):
        src = self._source_ids(tbl)
        year = tbl.column("publication_year")
        retracted = pc.fill_null(tbl.column("is_retracted"), False)

        keep = pc.and_(pc.is_in(src, value_set=want_sources),
                       pc.is_in(year, value_set=want_years))
        keep = pc.and_(keep, pc.invert(retracted))
        rows = pc.indices_nonzero(pc.fill_null(keep, False))
        if not len(rows):
            return None

        w = tbl.take(rows)
        w_src = [strip_id(x) for x in self._source_ids(w).to_pylist()]
        w_year = w.column("publication_year").to_pylist()

        pair_ok = [(s, y) in want_jy for s, y in zip(w_src, w_year)]
        if not any(pair_ok):
            return None
        idx = [i for i, ok in enumerate(pair_ok) if ok]
        w = w.take(idx)
        w_src = [w_src[i] for i in idx]
        w_year = [w_year[i] for i in idx]
        self.n_works += len(idx)

        auth_col = w.column("authorships")
        if isinstance(auth_col, pa.ChunkedArray):
            auth_col = auth_col.combine_chunks()
        parent = pc.list_parent_indices(auth_col)
        flat = auth_col.values
        ids = flat.field("author").field("id")

        not_treated = pc.invert(pc.fill_null(
            pc.is_in(ids, value_set=treated_arr), False))
        has_id = pc.invert(pc.is_null(ids))
        sel = pc.and_(not_treated, has_id)
        if not pc.any(sel).as_py():
            return None

        sel_ids = pc.filter(ids, sel).to_pylist()
        sel_parent = pc.filter(parent, sel).to_pylist()
        sel_pos = pc.filter(flat.field("author_position"), sel).to_pylist()
        sel_corr = pc.filter(flat.field("is_corresponding"), sel).to_pylist()
        sel_inst = pc.filter(flat.field("institutions"), sel).to_pylist()

        w_id = [strip_id(x) for x in w.column("id").to_pylist()]

        out = pd.DataFrame({
            "author_id": [strip_id(x) for x in sel_ids],
            "work_id": [w_id[p] for p in sel_parent],
            "source_id": [w_src[p] for p in sel_parent],
            "pub_year": [w_year[p] for p in sel_parent],
            "position": sel_pos,
            "is_corresponding": sel_corr,
            "country": [(i[0].get("country_code") if i else None)
                        for i in sel_inst],
        })

        self.writer.writerows(out[CANDIDATE_COLUMNS].to_dict("records"))
        self.f.flush()
        self.n_rows += len(out)
        self.authors.update(out.author_id)
        return None

    def close(self):
        self.f.close()


ex = Extractor(OUT_CANDIDATES, CANDIDATE_COLUMNS)
t0 = time.time()
try:
    snap.scan("works", SCAN_COLUMNS, ex.handle,
              limit_files=LIMIT_FILES, skip_files=SKIP_FILES,
              progress_every=200)
finally:
    ex.close()

print(f"\nelapsed {(time.time() - t0) / 60:.1f} min")
print(f"candidate works    {ex.n_works:,}")
print(f"candidate rows     {ex.n_rows:,}")
print(f"unique candidates  {len(ex.authors):,}")

scanning works: 2,446 files, 675.2 GiB on disk
  projecting 5 of 49 columns
  200/2,446 files | 0.0M rows | kept 0 | 31 MiB/s | eta 372m
  400/2,446 files | 46.7M rows | kept 0 | 203 MiB/s | eta 53m
  600/2,446 files | 81.3M rows | kept 0 | 184 MiB/s | eta 54m
  800/2,446 files | 119.0M rows | kept 0 | 178 MiB/s | eta 52m
  1,000/2,446 files | 155.3M rows | kept 0 | 173 MiB/s | eta 49m
  1,200/2,446 files | 192.2M rows | kept 0 | 169 MiB/s | eta 46m
  1,400/2,446 files | 230.6M rows | kept 0 | 167 MiB/s | eta 42m
  1,600/2,446 files | 271.4M rows | kept 0 | 157 MiB/s | eta 40m
  1,800/2,446 files | 326.2M rows | kept 0 | 153 MiB/s | eta 34m
  2,000/2,446 files | 380.3M rows | kept 0 | 154 MiB/s | eta 27m
  2,200/2,446 files | 434.0M rows | kept 0 | 136 MiB/s | eta 15m
  2,400/2,446 files | 493.9M rows | kept 0 | 128 MiB/s | eta 3m
  done: 510,372,821 rows scanned, 0 kept, 90.9 min

elapsed 90.9 min
candidate works    8,626,627
candidate rows     37,984,474
unique candidates  10,335,622

## The candidate pool

In [4]:
cand = pd.read_csv(OUT_CANDIDATES, low_memory=False)
print(f"rows               {len(cand):,}")
print(f"unique authors     {cand.author_id.nunique():,}")
print(f"distinct works     {cand.work_id.nunique():,}")
print(f"distinct journals  {cand.source_id.nunique():,}")

overlap = int(cand.author_id.astype(str).isin(treated).sum())
print(f"\ncandidate rows belonging to a treated author: {overlap:,}")
if overlap:
    print("  [!] the exclusion failed; these must be removed before matching")

print(f"\npapers per candidate author")
per = cand.groupby("author_id").work_id.nunique()
print(f"  median   {per.median():.0f}")
print(f"  mean     {per.mean():.1f}")
print(f"  maximum  {per.max():,}")

print(f"\nbyline position")
print(cand.position.value_counts(dropna=False).to_string())

print(f"\npublication year")
print(cand.pub_year.value_counts().sort_index().to_string())

rows               37,984,474
unique authors     10,335,622
distinct works     7,749,332
distinct journals  4,228

candidate rows belonging to a treated author: 0

papers per candidate author
  median   1
  mean     3.7
  maximum  3,253

byline position
position
middle    24026381
first      7557362
last       6400731

publication year
pub_year
2005     625729
2006     274544
2007     382956
2008     410428
2009     650630
2010     824440
2011     912426
2012    1619718
2013    1827418
2014    2681472
2015    3094031
2016    2933720
2017    3140460
2018    3466063
2019    4065275
2020    4787694
2021    4628804
2022    1658666


## Pool size against the matching requirement

Every treated author needs at least one candidate from a comparable
journal-year. Journal-years yielding few candidates are where treated authors
are most likely to go unmatched at Phase 6.

In [5]:
per_jy = cand.groupby(["source_id", "pub_year"]).author_id.nunique()
print(f"candidates per journal-year")
print(f"  median   {per_jy.median():.0f}")
print(f"  mean     {per_jy.mean():.1f}")
print(f"  minimum  {per_jy.min():,}")
print(f"  maximum  {per_jy.max():,}")
for k in [1, 5, 10, 25, 100]:
    n = int((per_jy < k).sum())
    print(f"  fewer than {k:>3}: {n:,} journal-years ({n / len(per_jy):.1%})")

ap = pd.read_csv(AUTHOR_PAPER, low_memory=False)
treated_jy = (ap.dropna(subset=["source_id", "pub_year"])
                .groupby(["source_id", "pub_year"]).author_id.nunique())
joined = (pd.DataFrame({"treated": treated_jy})
            .join(pd.DataFrame({"candidates": per_jy}), how="left")
            .fillna(0))
thin = joined[joined.candidates < joined.treated]
print(f"\njournal-years with fewer candidates than treated authors: "
      f"{len(thin):,} of {len(joined):,}")
print(f"  treated authors affected: {int(thin.treated.sum()):,}")

missing = joined[joined.candidates == 0]
print(f"\njournal-years with no candidates at all: {len(missing):,}")
print(f"  treated authors affected: {int(missing.treated.sum()):,}")

candidates per journal-year
  median   1078
  mean     3946.3
  minimum  1
  maximum  377,795
  fewer than   1: 0 journal-years (0.0%)
  fewer than   5: 10 journal-years (0.1%)
  fewer than  10: 23 journal-years (0.3%)
  fewer than  25: 89 journal-years (1.1%)
  fewer than 100: 676 journal-years (8.7%)

journal-years with fewer candidates than treated authors: 409 of 8,164
  treated authors affected: 1,973

journal-years with no candidates at all: 406
  treated authors affected: 1,941


## Coverage of the treated set

A treated author whose journal-year yielded no candidates cannot be matched
under exact matching on the retraction year, whatever the other covariates say.

In [6]:
have = set(zip(cand.source_id.astype(str), cand.pub_year.astype(int)))
ap_jy = ap.dropna(subset=["source_id", "pub_year"]).copy()
ap_jy["pub_year"] = ap_jy.pub_year.astype(int)
ap_jy["covered"] = [(s, y) in have for s, y
                    in zip(ap_jy.source_id.astype(str), ap_jy.pub_year)]

by_author = ap_jy.groupby("author_id").covered.any()
print(f"treated authors with at least one covered journal-year: "
      f"{int(by_author.sum()):,} of {len(by_author):,} "
      f"({by_author.mean():.1%})")

cov = (ap_jy.drop_duplicates("author_id")
            .set_index("author_id")[["category"]]
            .join(by_author.rename("covered")))
tab = cov.groupby("category").covered.agg(["size", "sum", "mean"])
tab.columns = ["authors", "covered", "rate"]
tab["rate"] = (tab["rate"] * 100).round(1)
print(f"\nby category")
print(tab.to_string())

treated authors with at least one covered journal-year: 61,543 of 62,637 (98.3%)

by category
                      authors  covered  rate
category                                    
AUTHOR_MISCONDUCT       33886    33010  97.4
EDITORIAL_COMPROMISE     8637     8593  99.5
ETHICS_VIOLATION          923      914  99.0
HONEST_ERROR            13129    13038  99.3
UNCLASSIFIED              468      460  98.3
UNCONFIRMED_CONCERNS     5594     5528  98.8
